# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` fields for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and create Dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps in understanding the data structure for targeted extraction and exploration.

All references to record sets and fields use their `@id` field.

In [ ]:
# List all record set @id's in the metadata.
# The Croissant metadata structure puts RecordSet objects in the `.record_sets` attribute (if present).

record_sets = []
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for record_set in metadata.record_sets:
        print(f"RecordSet @id: {record_set['@id']}, name: {record_set.get('name','--')}")
        record_sets.append(record_set['@id'])
else:
    print("No RecordSets found in metadata.")

# For demonstration (and in this dataset), if record_sets is empty, check for record sets via dataset.records (mlcroissant discovers them anyway).
if not record_sets:
    discovered = dataset.available_record_sets()
    record_sets = discovered
    print("Discovered record set IDs via .available_record_sets():")
    for rs in discovered:
        print(rs)
        print("Sample fields/columns for RecordSet ID", rs)
        for field in dataset.available_fields(record_set=rs):
            print("  Field @id:", field)

# Show example records for the first record set
if record_sets:
    example_record_set_id = record_sets[0]
    print(f"\nExample records from RecordSet @id: {example_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i > 2:  # Show only first 3 records
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id` values identified above.

In [ ]:
# Build DataFrames for each record set using their @id.
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

# For the next steps, select the main data record set ID
main_record_set_id = record_sets[0]
print(f"Main record set for analysis: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using field `@id`s. This includes filtering, normalizing, and grouping data using columns referenced by their `@id` from the Croissant schema.

Assume numeric fields exist in the main record set (e.g., `cr:MW` for molecular weight, or `cr:Abundance` for abundance measure). You may list the actual available field `@id`s using the following code cell.

In [ ]:
# List columns in the main DataFrame (should correspond to field @id values)
print(f"Fields (columns) available in {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())

# For demonstration, select a numeric field by @id (adapt these to actual field @ids from schema)
# Example candidates: 'cr:MW' (molecular weight), 'cr:Abundance', etc.
# Use whichever is present in your data:

candidate_numeric_fields = ['cr:MW', 'cr:MolecularWeight', 'cr:Abundance', 'cr:coverage', 'cr:peptide_count']
numeric_field = None
for field in candidate_numeric_fields:
    if field in dataframes[main_record_set_id].columns:
        numeric_field = field
        break
if not numeric_field:
    raise ValueError('No expected numeric field found. Please update candidate_numeric_fields list.')
print(f"Selected numeric field for analysis: {numeric_field}")

threshold = dataframes[main_record_set_id][numeric_field].mean()  # Dynamic threshold (mean)
# Remove NaN and filter records above mean
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field].fillna(0) > threshold].copy()
print(f"Filtered records where {numeric_field} > {threshold:0.2f} (mean):")
display(filtered_df.head())

# Normalize
filtered_df[numeric_field + '_normalized'] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
    filtered_df[numeric_field].std()
)
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

candidate_group_fields = ['cr:description', 'cr:gene_name', 'cr:sample', 'cr:group']
group_field = None
for field in candidate_group_fields:
    if field in dataframes[main_record_set_id].columns:
        group_field = field
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())
else:
    print('No group field found in the DataFrame columns.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and the normalized results, and show mean values grouped by the grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt

# Histogram of numeric field
plt.figure(figsize=(8, 4))
dataframes[main_record_set_id][numeric_field].dropna().hist(bins=30)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot for group (if group_field is found)
if group_field:
    plt.figure(figsize=(12, 6))
    filtered_df.boxplot(column=numeric_field, by=group_field, rot=90)
    plt.title(f'{numeric_field} by {group_field}')
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

# Scatterplot example: numeric_field vs normalized
if numeric_field + '_normalized' in filtered_df.columns:
    plt.figure(figsize=(6,4))
    plt.scatter(filtered_df[numeric_field], filtered_df[numeric_field + '_normalized'], alpha=0.5)
    plt.title(f'{numeric_field} vs {numeric_field}_normalized')
    plt.xlabel(numeric_field)
    plt.ylabel(numeric_field + '_normalized')
    plt.grid(True)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and visualize data from a Croissant-described dataset using the `mlcroissant` library. All references used schema `@id` fields for robust and reproducible analysis.

- We successfully loaded metadata and records, viewed the dataset's structure, and performed basic filtering and normalization.
- Columns selected for analysis and grouping were accessed via their unique `@id` field.
- The exploratory steps demonstrated how Croissant metadata enables direct and semantically meaningful data loading, processing, and analysis.

For further analysis, you may consult the dataset's documentation or schema for more advanced use cases and field-specific semantics.